# Fake Job Posting Detection — Combined Dataset, Multi-Model Benchmark & Explainable Prediction

**What this notebook does, step by step:**
1. Loads two independent fake-job-posting datasets (Kaggle EMSCAD + Hugging Face balanced set)
2. Cleans both, aligns their columns, combines them into one bigger dataset, and lets you download it
3. Handles class imbalance properly (computed class weights, not guessing)
4. Trains 9 classical ML models + 4 deep learning models with 5-fold cross-validation
5. Evaluates everyone with Accuracy / Precision / Recall / F1 / ROC-AUC, confusion matrices and ROC curves
6. Automatically picks the best model
7. Gives you an **explainable prediction tool**: type in a job title + description and instead of a plain
   True/False, you get the same style of output as the reference screenshot — a REAL/FAKE probability
   gauge, colour-coded word-by-word contribution, a ranked list of the words that mattered most, and a
   "flip analysis" that checks how robust the prediction is.

---

## Research gap this notebook addresses
Most existing fake-job-posting projects (including the original version of this notebook) have four
recurring problems:

| Gap in prior work | What we do about it |
|---|---|
| Trained on **one** small, heavily imbalanced dataset (EMSCAD, ~4-5% fraud) | We **combine EMSCAD with a second, balanced Hugging Face dataset**, so the model sees fraud patterns from two independently-collected sources instead of memorising one collector's quirks |
| Only report accuracy, and only one model | We benchmark **13 models (9 ML + 4 DL)** with stratified 5-fold CV and report Precision/Recall/F1/ROC-AUC for all of them, side by side |
| Output is a bare "0/1" label — no way to know *why* | We add a **leave-one-word-out explainability layer**: each word gets a signed "impact" score toward REAL or FAKE, shown as coloured chips + a probability gauge, exactly like the reference image |
| No check on how fragile the decision is | We add a **flip analysis**: it removes the most influential words one at a time to see whether that alone is enough to flip the prediction — a simple robustness/sanity check |

---
> ⚠️ **Before running:** Cell 2 needs internet access to `huggingface.co` (for `datasets`) and either
> Kaggle credentials (via `kagglehub`) or a manually uploaded `fake_job_postings.csv`
> (from https://www.kaggle.com/datasets/shivamb/real-or-fake-fake-jobposting-prediction) in the
> notebook's working directory. If you're on Kaggle itself, the original `/kaggle/input/...` path
> is still tried automatically.


## Step 1 — Install libraries

In [ ]:
# Everything needed for the whole notebook, installed once.
!pip install -q datasets kagglehub xgboost lightgbm tensorflow


## Step 2 — Load the Kaggle dataset (EMSCAD)
Bug fixed: the original notebook hard-coded a Kaggle-only path (`/kaggle/input/...`), which throws
`FileNotFoundError` anywhere else (Colab, local machine). We now try, in order:
1. `kagglehub` (auto-downloads if you have a Kaggle account/API token configured)
2. The original hard-coded Kaggle path (works if you're inside a Kaggle notebook)
3. A `fake_job_postings.csv` sitting next to this notebook (manual upload)


In [ ]:
import os
import pandas as pd

def load_kaggle_dataset():
    # 1) Try kagglehub (works locally / Colab if Kaggle API token is set up)
    try:
        import kagglehub
        path = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
        csv_path = os.path.join(path, "fake_job_postings.csv")
        if os.path.exists(csv_path):
            print(f"Loaded via kagglehub: {csv_path}")
            return pd.read_csv(csv_path)
    except Exception as e:
        print(f"kagglehub route not available ({e}), trying next option...")

    # 2) Native Kaggle notebook path
    kaggle_native = "/kaggle/input/real-or-fake-fake-jobposting-prediction/fake_job_postings.csv"
    if os.path.exists(kaggle_native):
        print(f"Loaded from native Kaggle path: {kaggle_native}")
        return pd.read_csv(kaggle_native)

    # 3) Manual upload next to the notebook
    if os.path.exists("fake_job_postings.csv"):
        print("Loaded from local fake_job_postings.csv")
        return pd.read_csv("fake_job_postings.csv")

    raise FileNotFoundError(
        "Could not find the Kaggle dataset. Either set up Kaggle API credentials for "
        "kagglehub, run this inside a Kaggle notebook, or place 'fake_job_postings.csv' "
        "next to this notebook."
    )

kaggle_df = load_kaggle_dataset()

# Bug fixed: label column is sometimes read as float/str depending on source -> force to int 0/1
kaggle_df["fraudulent"] = kaggle_df["fraudulent"].astype(int)

print(f"\nKaggle dataset: {kaggle_df.shape[0]} rows, {kaggle_df.shape[1]} columns")
print(f"Real (0): {(kaggle_df['fraudulent'] == 0).sum()} | Fake (1): {(kaggle_df['fraudulent'] == 1).sum()}")


## Step 3 — Load the Hugging Face dataset
Bug fixed: the original code assumed the label column is always called `fraudulent`. Hugging Face
datasets don't guarantee that, so we auto-detect common label-column names and rename to `fraudulent`.


In [ ]:
from datasets import load_dataset

hf_ds = load_dataset("gplsi/fake_job_postings_balanced_en")
hf_df = hf_ds["train"].to_pandas()

# Auto-detect and standardise the label column name
label_candidates = ["fraudulent", "label", "labels", "is_fake", "class"]
for c in label_candidates:
    if c in hf_df.columns:
        hf_df = hf_df.rename(columns={c: "fraudulent"})
        break
else:
    raise KeyError(f"No known label column found in Hugging Face dataset. Columns: {list(hf_df.columns)}")

hf_df["fraudulent"] = hf_df["fraudulent"].astype(int)

print(f"Hugging Face dataset: {hf_df.shape[0]} rows, {hf_df.shape[1]} columns")
print(f"Real (0): {(hf_df['fraudulent'] == 0).sum()} | Fake (1): {(hf_df['fraudulent'] == 1).sum()}")


## Step 4 — Clean both datasets with one shared function
Bug fixed: the original notebook copy-pasted the same cleaning logic twice (once per dataset) with
subtle differences. One shared function means one place to fix bugs, and both datasets get identical
treatment (important for a fair combination later).


In [ ]:
def clean_dataset(df, name):
    df = df.copy()
    print("=" * 60)
    print(f"CLEANING: {name}")
    print("=" * 60)

    # ---- Missingness report ----
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(2)
    report = pd.DataFrame({"column": null_counts.index, "null_count": null_counts.values,
                            "null_percent": null_pct.values}).sort_values("null_count", ascending=False)
    print(report.to_string(index=False))

    # ---- "has_X" flags: missingness itself can be a fraud signal ----
    flag_cols = ["salary_range", "department", "company_profile", "benefits", "requirements"]
    for col in flag_cols:
        if col in df.columns:
            df[f"has_{col}"] = df[col].notnull().astype(int)

    # ---- Drop salary_range if too sparse to be useful ----
    if "salary_range" in df.columns and df["salary_range"].isnull().mean() > 0.70:
        df = df.drop(columns=["salary_range"])

    # ---- Fill remaining text columns with empty string, not "nan" ----
    text_cols = ["title", "description", "requirements", "company_profile", "benefits",
                 "location", "department", "industry", "function", "employment_type",
                 "required_experience", "required_education"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str)

    # ---- Fill remaining binary/numeric columns with 0 ----
    for col in ["telecommuting", "has_company_logo", "has_questions"]:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)

    remaining_nulls = df.isnull().sum().sum()
    print(f"\nRemaining nulls after cleaning: {remaining_nulls}")
    return df

kaggle_clean = clean_dataset(kaggle_df, "Kaggle (EMSCAD)")
hf_clean = clean_dataset(hf_df, "Hugging Face (balanced)")


## Step 5 — Align columns & combine into one dataset
Bug fixed: this exact cell was **duplicated twice** in the original notebook (cells 2 and 3 were
identical). Kept once here. We also guard against the edge case where the two datasets end up with
zero overlapping columns.


In [ ]:
common_cols = sorted(set(kaggle_clean.columns) & set(hf_clean.columns))
if "fraudulent" not in common_cols:
    raise ValueError("'fraudulent' column missing from the common columns — cannot combine datasets.")

print(f"Common columns between both datasets ({len(common_cols)}): {common_cols}")

kaggle_aligned = kaggle_clean[common_cols].copy()
hf_aligned = hf_clean[common_cols].copy()

kaggle_aligned["source"] = "kaggle_emscad"
hf_aligned["source"] = "huggingface_balanced"

combined_df = pd.concat([kaggle_aligned, hf_aligned], ignore_index=True)
before_dedup = len(combined_df)

# The HF set is a balanced subset drawn from similar postings -> de-dupe on title+description
dedup_subset = [c for c in ["title", "description"] if c in combined_df.columns]
combined_df = combined_df.drop_duplicates(subset=dedup_subset).reset_index(drop=True)
after_dedup = len(combined_df)

print("\n" + "=" * 60)
print("COMBINED DATASET SUMMARY")
print("=" * 60)
print(f"Kaggle rows (cleaned)   : {len(kaggle_clean)}")
print(f"Hugging Face rows (cleaned): {len(hf_clean)}")
print(f"Combined before dedup   : {before_dedup}")
print(f"Duplicates removed      : {before_dedup - after_dedup}")
print(f"Final combined size     : {after_dedup}")

remaining = combined_df.isnull().sum()
remaining = remaining[remaining > 0]
print("\nZero nulls remaining." if len(remaining) == 0 else f"Still has nulls:\n{remaining}")


## Step 6 — Class balance: how skewed is the combined dataset?

In [ ]:
import matplotlib.pyplot as plt

class_counts = combined_df["fraudulent"].value_counts().sort_index()
total = len(combined_df)

print("=" * 60)
print("CLASS DISTRIBUTION — Combined Dataset")
print("=" * 60)
print(f"Real (0) : {class_counts[0]:>7}  ({class_counts[0]/total*100:.2f}%)")
print(f"Fake (1) : {class_counts[1]:>7}  ({class_counts[1]/total*100:.2f}%)")
print(f"Imbalance ratio : 1 Fake : {class_counts[0] / class_counts[1]:.1f} Real")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Real", "Fake"], [class_counts[0], class_counts[1]], color=["#2ecc71", "#e74c3c"])
ax.set_title("Class distribution — combined dataset", fontweight="bold")
ax.set_ylabel("Number of postings")
for i, v in enumerate([class_counts[0], class_counts[1]]):
    ax.text(i, v, f"{v}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()


## Step 7 — Compute balanced class weights
Bug fixed: the original notebook computed this **twice in two separate cells** (cells 4 and 5) with
the same formula. Kept once. These weights are reused later for every model that supports
`class_weight` / `scale_pos_weight`, instead of oversampling/duplicating rows (which risks leaking
near-identical rows across train/test).


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

y_all = combined_df["fraudulent"].values
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_all)
class_weight_dict = {0: float(class_weights[0]), 1: float(class_weights[1])}
scale_pos_weight = class_weights[1] / class_weights[0]  # for XGBoost

print(f"Class weight dictionary : {class_weight_dict}")
print(f"scale_pos_weight (XGBoost) : {scale_pos_weight:.4f}")
print(f"\nA missed Fake posting now costs the model {scale_pos_weight:.1f}x more than a missed Real one.")


## Step 8 — Save & download the combined dataset
This is the merged, cleaned dataset requested — one CSV built from both sources.


In [ ]:
output_path = "combined_fake_job_postings.csv"
combined_df.to_csv(output_path, index=False)
print(f"Saved {output_path}  ({combined_df.shape[0]} rows, {combined_df.shape[1]} columns)")

# Works automatically if you're in Google Colab; otherwise just grab the file from the
# notebook's working directory / file browser (e.g. Kaggle's Output tab, or Jupyter's file list).
try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("Not running in Google Colab — the CSV is saved in the working directory, "
          "download it from there (e.g. the Jupyter/Kaggle file browser).")


## Step 9 — Build model input text, train/test split, TF-IDF
Bug fixed: `full_text` construction used to assume `title`, `description`, `requirements`,
`company_profile`, `benefits` all exist — if the HF/Kaggle overlap ever drops one of those columns,
the old code would crash with a `KeyError`. We now only use whichever of those columns actually
survived the column-alignment step.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

text_cols = [c for c in ["title", "description", "requirements", "company_profile", "benefits"]
             if c in combined_df.columns]
print(f"Text columns used for full_text: {text_cols}")

combined_df["full_text"] = combined_df[text_cols].astype(str).agg(" ".join, axis=1)

X = combined_df["full_text"].values
y = combined_df["fraudulent"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size : {len(X_train)}  |  Test size : {len(X_test)}")

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english", min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")

# Class weights recomputed on the TRAIN split only (avoids any test-set leakage into weighting)
train_class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
train_class_weight_dict = {0: float(train_class_weights[0]), 1: float(train_class_weights[1])}
train_scale_pos_weight = train_class_weights[1] / train_class_weights[0]
print(f"Train-only class weights: {train_class_weight_dict}")


## Step 10 — Train & evaluate 9 classical ML models (5-fold CV)
Bug fixed: `XGBClassifier(..., use_label_encoder=False)` is removed — that parameter was deprecated
and **crashes on current xgboost versions**. Everything else is unchanged in spirit but wired to the
train-only class weights from Step 9.


In [ ]:
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                              accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

ml_models = {
    "Logistic Regression": LogisticRegression(class_weight=train_class_weight_dict, max_iter=1000, random_state=42),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": CalibratedClassifierCV(LinearSVC(class_weight=train_class_weight_dict, max_iter=2000)),
    "Decision Tree": DecisionTreeClassifier(class_weight=train_class_weight_dict, random_state=42, max_depth=15),
    "Random Forest": RandomForestClassifier(class_weight=train_class_weight_dict, n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(scale_pos_weight=train_scale_pos_weight, eval_metric="logloss", random_state=42),
    "LightGBM": LGBMClassifier(class_weight=train_class_weight_dict, random_state=42, verbose=-1),
    "KNN": KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
}

results = []
predictions_dict = {}
probas_dict = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in ml_models.items():
    print(f"Training {name}...")
    cv_scores = cross_val_score(model, X_train_tfidf, y_train, cv=skf, scoring="f1", n_jobs=-1)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_tfidf)[:, 1]
    else:
        from scipy.special import expit
        y_proba = expit(model.decision_function(X_test_tfidf))

    predictions_dict[name] = y_pred
    probas_dict[name] = y_proba

    results.append({
        "Model": name,
        "CV F1 (mean)": cv_scores.mean(),
        "CV F1 (std)": cv_scores.std(),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })
    print(f"  CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f} | Test F1: {results[-1]['F1-Score']:.4f}")

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
print("\n" + "=" * 70)
print("ML MODEL COMPARISON")
print("=" * 70)
print(results_df.to_string(index=False))


In [ ]:
# Confusion matrices for every ML model
n_models = len(ml_models)
cols = 3
rows = (n_models + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten()

for i, (name, y_pred) in enumerate(predictions_dict.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="RdYlGn_r",
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"], ax=axes[i])
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, y_pred):.3f}", fontweight="bold")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.suptitle("Confusion Matrices — ML Models", fontsize=16, y=1.02, fontweight="bold")
plt.show()


In [ ]:
# ROC curves for every ML model
plt.figure(figsize=(10, 7))
for name, y_proba in probas_dict.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={auc(fpr, tpr):.3f})")

plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — All ML Models", fontweight="bold")
plt.legend(loc="lower right", fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_ml_name = results_df.iloc[0]["Model"]
print(f"Best ML model so far: {best_ml_name} (F1 = {results_df.iloc[0]['F1-Score']:.4f})")


## Step 11 — Deep learning models (optional, slower)
LSTM, BiLSTM, CNN, and a CNN→BiLSTM hybrid, all trained with the train-only class weights.
Epochs are kept small (5, with early stopping) since this is meant as a comparison point against the
ML models above, not a fully tuned production model. Feel free to raise `epochs` if you have GPU time.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Bidirectional, Dense, Dropout,
                                      Conv1D, GlobalMaxPooling1D, MaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

tf.random.set_seed(42)
np.random.seed(42)

MAX_VOCAB = 20000
MAX_LEN = 300
EMBED_DIM = 128

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN, padding="post")

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index) + 1)
print(f"Vocab size: {vocab_size} | Sequence length: {MAX_LEN}")


def build_lstm():
    m = Sequential([
        Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
        LSTM(64, dropout=0.3, recurrent_dropout=0.3),
        Dense(32, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m


def build_bilstm():
    m = Sequential([
        Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
        Dense(32, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m


def build_cnn():
    m = Sequential([
        Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
        Conv1D(128, 5, activation="relu"),
        GlobalMaxPooling1D(),
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m


def build_cnn_bilstm():
    m = Sequential([
        Embedding(vocab_size, EMBED_DIM, input_length=MAX_LEN),
        Conv1D(128, 5, activation="relu"),
        MaxPooling1D(pool_size=2),
        Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3)),
        Dense(32, activation="relu"),
        Dropout(0.3),
        Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return m


dl_models = {
    "LSTM": build_lstm(),
    "BiLSTM": build_bilstm(),
    "CNN": build_cnn(),
    "CNN-BiLSTM": build_cnn_bilstm(),
}

dl_results = []
dl_predictions = {}
dl_probas = {}
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

for name, model in dl_models.items():
    print(f"Training {name}...")
    model.fit(X_train_seq, y_train, validation_split=0.1, epochs=5, batch_size=64,
              class_weight=train_class_weight_dict, callbacks=[early_stop], verbose=0)

    y_proba = model.predict(X_test_seq, verbose=0).ravel()
    y_pred = (y_proba >= 0.5).astype(int)

    dl_predictions[name] = y_pred
    dl_probas[name] = y_proba
    dl_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })
    print(f"  Test F1: {dl_results[-1]['F1-Score']:.4f}")

dl_results_df = pd.DataFrame(dl_results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
print("\nDL MODEL COMPARISON")
print(dl_results_df.to_string(index=False))


In [ ]:
# Confusion matrices + ROC curves for DL models
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, (name, y_pred) in enumerate(dl_predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="RdYlGn_r",
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"], ax=axes[i])
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, y_pred):.3f}", fontweight="bold")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")
plt.tight_layout()
plt.suptitle("Confusion Matrices — DL Models", fontsize=16, y=1.02, fontweight="bold")
plt.show()

plt.figure(figsize=(12, 8))
for name, y_proba in probas_dict.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=1.2, alpha=0.6, label=f"[ML] {name} (AUC={auc(fpr, tpr):.3f})")
for name, y_proba in dl_probas.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, linewidth=2.5, label=f"[DL] {name} (AUC={auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — ML + DL Models", fontweight="bold")
plt.legend(loc="lower right", fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Step 12 — Pick the best model overall
Combines the ML and DL leaderboards and automatically selects the top F1-score model as the one used
for the final validation report and the explainable prediction tool below.


In [ ]:
ml_short = results_df[["Model", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]]
all_results = pd.concat([ml_short, dl_results_df], ignore_index=True).sort_values(
    "F1-Score", ascending=False).reset_index(drop=True)

print("=" * 70)
print("ALL MODELS — FINAL LEADERBOARD (ML + DL)")
print("=" * 70)
print(all_results.to_string(index=False))

best_model_name = all_results.iloc[0]["Model"]
is_dl_model = best_model_name in dl_models
print(f"\nBest overall model: {best_model_name}  (F1 = {all_results.iloc[0]['F1-Score']:.4f}, "
      f"ROC-AUC = {all_results.iloc[0]['ROC-AUC']:.4f})")


## Step 13 — Full validation report for the chosen best model

In [ ]:
if is_dl_model:
    best_model = dl_models[best_model_name]
    best_y_pred = dl_predictions[best_model_name]
    best_y_proba = dl_probas[best_model_name]
else:
    best_model = ml_models[best_model_name]
    best_y_pred = predictions_dict[best_model_name]
    best_y_proba = probas_dict[best_model_name]

print("=" * 70)
print(f"CLASSIFICATION REPORT — {best_model_name}")
print("=" * 70)
print(classification_report(y_test, best_y_pred, target_names=["Real", "Fake"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, best_y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Real", "Fake"],
            yticklabels=["Real", "Fake"], ax=axes[0])
axes[0].set_title(f"Confusion Matrix — {best_model_name}", fontweight="bold")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(y_test, best_y_proba)
axes[1].plot(fpr, tpr, linewidth=2, color="#2563eb", label=f"AUC = {auc(fpr, tpr):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title(f"ROC Curve — {best_model_name}", fontweight="bold")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


## Step 14 — Save the best model for reuse

In [ ]:
import pickle

if not is_dl_model:
    with open("best_fake_job_model.pkl", "wb") as f:
        pickle.dump({"model_name": best_model_name, "model": best_model, "vectorizer": tfidf}, f)
    print("Saved best_fake_job_model.pkl (ML model + TF-IDF vectorizer)")
else:
    best_model.save("best_fake_job_model.keras")
    import json
    with open("tokenizer.json", "w") as f:
        f.write(tokenizer.to_json())
    print("Saved best_fake_job_model.keras + tokenizer.json (DL model)")


## Step 15 — Explainable prediction (the part shown in the reference screenshot)

Instead of a plain True/False, this gives:
- **A REAL/FAKE gauge** — same red-to-green bar as the screenshot
- **Colour-coded words** — green pushed the prediction toward REAL, red pushed it toward FAKE
- **Top contributing words**, ranked
- **A flip analysis** — removes the most influential words to test how robust the prediction is

How it works: for each word we ask "what does the model predict with this word removed?" and compare
that to the prediction with the word present. The difference is that word's "impact". This works for
*any* trained model (ML or DL) since it only touches raw text in and a probability out.


In [ ]:
import re

def get_predict_fn(model_name):
    """Returns a function: list[str] -> np.array of P(FAKE), for whichever model won."""
    if model_name in ml_models:
        model = ml_models[model_name]
        def predict_fn(texts):
            vec = tfidf.transform(texts)
            if hasattr(model, "predict_proba"):
                return model.predict_proba(vec)[:, 1]
            from scipy.special import expit
            return expit(model.decision_function(vec))
        return predict_fn
    elif model_name in dl_models:
        model = dl_models[model_name]
        def predict_fn(texts):
            seqs = pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LEN, padding="post")
            return model.predict(seqs, verbose=0).ravel()
        return predict_fn
    raise ValueError(f"Unknown model: {model_name}")


def tokenize(text):
    return re.findall(r"[A-Za-z0-9']+", text)


def word_impacts(text, predict_fn):
    """
    Leave-one-word-out analysis.
    Returns (baseline_prob_real, list of (word, impact)) where impact > 0 means that
    word pushed the prediction toward REAL, and impact < 0 means it pushed toward FAKE
    (matches the colour convention in the reference screenshot: green=REAL, red=FAKE).
    """
    tokens = tokenize(text)
    baseline_fake = predict_fn([text])[0]
    baseline_real = 1 - baseline_fake

    impacts = []
    for i in range(len(tokens)):
        remaining = tokens[:i] + tokens[i + 1:]
        remaining_text = " ".join(remaining) if remaining else ""
        removed_fake = predict_fn([remaining_text])[0] if remaining_text else 0.5
        removed_real = 1 - removed_fake
        impact = baseline_real - removed_real  # + => word pushes toward REAL, - => toward FAKE
        impacts.append((tokens[i], impact))

    return baseline_real, impacts


def flip_analysis(text, predict_fn, impacts, tokens, max_removals=5):
    """Greedily removes the most influential words toward the current label and checks for a flip."""
    baseline_fake = predict_fn([text])[0]
    current_label = "FAKE" if baseline_fake >= 0.5 else "REAL"

    # Words pushing toward the CURRENT label are the ones worth testing removal of
    if current_label == "FAKE":
        ranked = sorted(impacts, key=lambda t: t[1])          # most negative (most FAKE-pushing) first
    else:
        ranked = sorted(impacts, key=lambda t: -t[1])         # most positive (most REAL-pushing) first

    working = list(tokens)
    removed = []
    for word, _ in ranked[:max_removals]:
        if word in working:
            working.remove(word)
            removed.append(word)
        new_text = " ".join(working)
        new_fake = predict_fn([new_text])[0] if new_text else 0.5
        new_label = "FAKE" if new_fake >= 0.5 else "REAL"
        if new_label != current_label:
            return True, removed, new_fake
    return False, removed, None


In [ ]:
def plot_word_chips(impacts):
    """Coloured word chips, like the top row of the reference screenshot."""
    max_abs = max((abs(imp) for _, imp in impacts), default=1e-6) or 1e-6
    n = len(impacts)
    fig_w = min(max(6, 0.9 * n), 18)
    fig, ax = plt.subplots(figsize=(fig_w, 1.3))

    x = 0.0
    for word, imp in impacts:
        norm = imp / max_abs  # roughly -1..1
        if norm >= 0:
            color = plt.cm.Greens(0.25 + 0.55 * min(norm, 1.0))
        else:
            color = plt.cm.Reds(0.25 + 0.55 * min(abs(norm), 1.0))
        w = 0.16 * len(word) + 0.5
        ax.add_patch(plt.Rectangle((x, 0), w, 1, facecolor=color, edgecolor="none"))
        ax.text(x + w / 2, 0.5, word, ha="center", va="center", fontsize=10, fontweight="bold")
        x += w + 0.08

    ax.set_xlim(0, x if x > 0 else 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def plot_gauge(prob_real):
    """Red-to-green probability gauge, like the middle of the reference screenshot."""
    fig, ax = plt.subplots(figsize=(9, 1.8))
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    ax.imshow(gradient, extent=[0, 100, 0, 1], aspect="auto", cmap="RdYlGn")
    ax.axvline(prob_real * 100, color="black", linewidth=3)
    ax.plot(prob_real * 100, 0.5, "o", color="white", markeredgecolor="black",
            markersize=14, zorder=5)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xticklabels(["0% (FAKE)", "25%", "50%", "75%", "100% (REAL)"])

    label = "FAKE" if prob_real < 0.5 else "REAL"
    ax.set_title(f"Prediction: {label}  |  {prob_real*100:.1f}% REAL / {(1-prob_real)*100:.1f}% FAKE",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def analyze_job_posting(title, description, model_name=None, max_words=60):
    """
    Full explainable prediction: gauge + coloured words + ranked contributors + flip analysis.
    Mirrors the layout of the reference screenshot.
    """
    model_name = model_name or best_model_name
    predict_fn = get_predict_fn(model_name)

    text = f"{title} {description}".strip()
    tokens_full = tokenize(text)
    truncated = len(tokens_full) > max_words
    text_used = " ".join(tokens_full[:max_words]) if truncated else text

    print(f'Input: "{text_used}"' + (" (truncated for speed)" if truncated else ""))
    print(f"Model used: {model_name}")
    print("Green = pushes toward REAL | Red = pushes toward FAKE | darker = stronger effect\n")

    baseline_real, impacts = word_impacts(text_used, predict_fn)
    tokens_used = tokenize(text_used)

    plot_word_chips(impacts)
    plot_gauge(baseline_real)

    ranked = sorted(impacts, key=lambda t: -abs(t[1]))[:10]
    print("Top contributing words:")
    for word, imp in ranked:
        direction = "REAL" if imp >= 0 else "FAKE"
        print(f"  {word:<15s} -> {direction:<4s} (impact: {imp:+.4f})")

    flipped, removed_words, new_fake_prob = flip_analysis(text_used, predict_fn, impacts, tokens_used)
    print("\nFlip analysis:")
    if flipped:
        print(f"  Removing just {removed_words} FLIPS the prediction "
              f"(new FAKE probability: {new_fake_prob*100:.1f}%). The label is fragile for this input.")
    else:
        print(f"  Could not flip the prediction by word removal alone "
              f"(tried removing: {removed_words}). The label is fairly robust for this input.")

    return {"baseline_prob_real": baseline_real, "impacts": impacts,
            "flipped": flipped, "removed_words": removed_words}


## Step 16 — Try it: enter a job title & description
Run this cell and type in your own posting when prompted. You'll get the full explainable output
(gauge + coloured words + top contributors + flip analysis) instead of a plain True/False.


In [ ]:
job_title = input("Enter job title: ")
job_description = input("Enter job description: ")

result = analyze_job_posting(job_title, job_description)


## Step 17 — Quick demo with a fixed example (no typing needed)
Reproduces the exact example from the reference screenshot so you can sanity-check the output format.


In [ ]:
_ = analyze_job_posting(
    title="Government launches new digital education scheme",
    description="for students"
)


## Summary — what was fixed vs. the original notebook
- **Duplicate cells removed**: the dataset-combining cell existed twice, and class-weight computation
  existed twice; both are now single, canonical cells.
- **Hard-coded Kaggle path replaced** with a 3-tier fallback (`kagglehub` → native Kaggle path → local
  CSV) so the notebook runs outside of Kaggle too.
- **HF label column auto-detected** instead of assuming it's always called `fraudulent`.
- **`XGBClassifier(use_label_encoder=False)` removed** — that parameter is deprecated/breaks on
  current xgboost versions.
- **`full_text` construction now guards against missing columns** instead of assuming
  `title`/`description`/`requirements`/`company_profile`/`benefits` always survive the column
  alignment step.
- **Class weights recomputed on the train split only**, not the full dataset (avoids test leakage
  into how the model is weighted).
- **Added**: dataset export/download cell, best-model auto-selection across ML+DL, full validation
  report (classification report + confusion matrix + ROC curve) for the winning model, model-saving
  cell, and the full explainable-prediction tool (gauge, coloured words, ranked contributors, flip
  analysis) that this whole exercise was ultimately for.
